# Bước 2: Data Collection, Understanding & Preparation
## Kiểm Toán Chất Lượng Dữ Liệu Theo Khung DAMA-DMBOK & Lưu Vết (Audit Trail)
### Phân Tích Hành Vi Mua Hàng Của Khách Hàng Trong Thương Mại Điện Tử (Tập Dữ Liệu Gốc: 500.000 Bản Ghi)
#### Đề án Môn học: Applied Data Analytics (ADY201m) — ĐH FPT
---


## 2.0 Tuyên ngôn Các Nguyên tắc Quản trị Dữ liệu (Data Governance Principles)

Nhằm đảm bảo tính khoa học, minh bạch và khả năng tái lập trong thực tế doanh nghiệp, toàn bộ quy trình tiền xử lý dữ liệu của dự án được xây dựng dựa trên 3 trụ cột cốt lõi:

1. **Khung Đánh Giá Toàn Diện (DAMA-DMBOK 6 Chiều):**
   Đánh giá và kiểm toán chất lượng dữ liệu có hệ thống theo 6 chiều chuẩn mực quốc tế:
   - *Completeness (Đầy đủ):* Không chứa ô trống (Null/NaN) trên toàn bộ 500.000 bản ghi.
   - *Uniqueness (Duy nhất):* Khử trùng lặp (Deduplication), đảm bảo mỗi bản ghi là một phiên tương tác độc nhất.
   - *Validity (Hợp lệ):* Giá trị biến số phải nằm đúng miền logic thực tế (không chấp nhận số âm ở biến đếm hoặc biến thời gian).
   - *Consistency (Nhất quán):* Logic nghiệp vụ giữa các thuộc tính không mâu thuẫn (ví dụ: ngày mua gần nhất không thể lớn hơn thâm niên tài khoản).
   - *Accuracy (Chính xác):* Giữ nguyên độ chính xác số học (Round-trip precision), không làm tròn sai lệch kiểu số.
   - *Timeliness (Kịp thời):* Phản ánh đúng thời điểm phát sinh hành vi, kiểm soát chặt chẽ mốc thời gian để loại bỏ hoàn toàn rủi ro rò rỉ dữ liệu (No Data Leakage).

2. **Kỹ Thuật Lưu Vết Kiểm Toán (Audit Trail & Reproducibility):**
   - *Tính bất biến của dữ liệu gốc:* Tuyệt đối không can thiệp hay ghi đè lên file dữ liệu ban đầu. Kiểm định tính toàn vẹn cấu trúc (500.000 dòng × 17 cột) bằng bảng cờ lỗi `issues` độc lập.
   - *Phân tách có lưu vết:* Tách 100% bản ghi vi phạm (67.744 dòng) trên bảng cờ lỗi `issues` độc lập, đảm bảo mọi thao tác đều có thể truy vết và kiểm chứng.

3. **Bảo Toàn Tính Đại Diện & Chân Thực Của Thị Trường:**
   - Phân định rạch ròi giữa **dữ liệu sai hỏng kỹ thuật/logic** (7 vấn đề cần bóc tách) và **biến thiên tự nhiên của khách hàng thật** (như nhóm khách hàng có thu nhập cao hoặc gắn bó lâu năm).
   - Dự án bảo toàn trọn vẹn các quan sát thực tế này, không sử dụng công thức toán cắt gọt cơ học (Trimming hay Winsorization theo IQR) làm méo mó chân dung khách hàng thật.

## 2.1 Import Thư viện & Thiết lập Môi trường Làm việc

**Mục tiêu:** Nạp thư viện phân tích dữ liệu cốt lõi (`pandas`) và tự động thiết lập đường dẫn truy xuất tệp dữ liệu, đảm bảo code chạy tương thích và ổn định ở mọi môi trường thực thi.


In [1]:
import pandas as pd

DATA_DIR = '../Data'

RAW_PATH = f'{DATA_DIR}/customerData_500k.csv'
CLEAN_PATH = f'{DATA_DIR}/customerData_clean.csv'

## 2.2 Nạp Dữ Liệu & Khảo Sát Cấu Trúc Tổng Quan (Shape & Dtypes)

* **Mục tiêu:** Nạp tệp dữ liệu thô customerData_500k.csv, xác thực kích thước tổng thể (số dòng, số cột) và kiểm tra kiểu dữ liệu của 17 thuộc tính.

In [2]:
raw0 = pd.read_csv(RAW_PATH)
print('Kích thước (dòng, cột):', raw0.shape)
print(raw0.dtypes)

Kích thước (dòng, cột): (500000, 17)
Age                       int64
AnnualIncome            float64
NumberOfPurchases         int64
TimeSpentOnWebsite      float64
CustomerTenureYears     float64
LastPurchaseDaysAgo       int64
Gender                      str
ProductCategory             str
PreferredDevice             str
Region                      str
ReferralSource              str
CustomerSegment             str
LoyaltyProgram            int64
DiscountsAvailed          int64
SessionCount              int64
CustomerSatisfaction    float64
PurchaseStatus            int64
dtype: object


**Kết luận:**
* Bảng dữ liệu thô gồm **500.000 dòng × 17 cột**.
* Phân bố kiểu dữ liệu: 8 thuộc tính số nguyên (int64), 3 thuộc tính số thực (




loat64) và 6 thuộc tính phân loại dạng chuỗi (object). Các thuộc tính đều có kiểu dữ liệu phù hợp với ý nghĩa nghiệp vụ ban đầu, chưa phát hiện trường hợp số bị lưu nhầm dạng text.

### 2.2.1 Kiểm Toán & Xử Lý Tính Đầy Đủ (Completeness - Missing Value Handling)

**Mục tiêu:** Rà soát dữ liệu khuyết thiếu trên 17 thuộc tính và lựa chọn phương án xử lý bảo toàn cấu trúc dữ liệu mà không làm lệch phân phối.

In [3]:
print("Số ô khuyết thiếu:")
print(raw0.isnull().sum())

Số ô khuyết thiếu:
Age                       0
AnnualIncome            987
NumberOfPurchases         0
TimeSpentOnWebsite        0
CustomerTenureYears       0
LastPurchaseDaysAgo       0
Gender                    0
ProductCategory           0
PreferredDevice           0
Region                    0
ReferralSource            0
CustomerSegment           0
LoyaltyProgram            0
DiscountsAvailed          0
SessionCount              0
CustomerSatisfaction    575
PurchaseStatus            0
dtype: int64


* **Tỷ lệ thiếu rất nhỏ (0,31% toàn bảng):** Chỉ có `AnnualIncome` (987 ô, 0,20%) và `CustomerSatisfaction` (575 ô, 0,12%) bị khuyết. Chọn phương án điền khuyết (imputation) thay vì xóa dòng để bảo toàn trọn vẹn 500.000 bản ghi.
* **Cân nhắc lựa chọn giữa Mean và Median:** Khảo sát thống kê phân phối của 2 biến khuyết thiếu để chọn giá trị điền phù hợp:

In [4]:
raw0[['AnnualIncome', 'CustomerSatisfaction']].describe().round(2)

,AnnualIncome,CustomerSatisfaction
count,499013.00,499425.00
mean,85068.57,3.22
std,39589.57,0.83
min,11966.39,1.00
25%,51991.31,3.00
50%,83750.20,3.00
75%,116550.05,4.00
max,204178.29,5.00


**Căn cứ chọn Median:**
* **`AnnualIncome`:** Mean (~85.069 USD) và Median (~83.750 USD) chênh lệch nhỏ (~1,5%); chọn **Median** để giữ nguyên phân phối thực tế và tránh nhạy cảm với các giá trị cực trị.
* **`CustomerSatisfaction`:** Thang đo 1–5 sao chỉ nhận giá trị nguyên. Mean lẻ (3,22 sao) không phản ánh thực tế khảo sát; chọn **Median = 3.0 sao** để giữ đúng thang điểm nguyên và đại diện mức đánh giá trung tính khi khách không điền.

In [5]:
med_income = raw0['AnnualIncome'].median()
med_sat = raw0['CustomerSatisfaction'].median()

raw0['AnnualIncome'] = raw0['AnnualIncome'].fillna(med_income)
raw0['CustomerSatisfaction'] = raw0['CustomerSatisfaction'].fillna(med_sat)

print("Xử lý hoàn tất:")
print(f"- Điền trung vị AnnualIncome: {med_income:,.0f} USD")
print(f"- Điền trung vị CustomerSatisfaction: {med_sat:.1f} sao")
print(f"- Kiểm tra lại số ô thiếu toàn bảng: {raw0.isnull().sum().sum()} ô")

Xử lý hoàn tất:
- Điền trung vị AnnualIncome: 83,750 USD
- Điền trung vị CustomerSatisfaction: 3.0 sao
- Kiểm tra lại số ô thiếu toàn bảng: 0 ô


**Kết luận:** Dữ liệu đạt tính đầy đủ (Completeness) 100% với 0 ô khuyết thiếu, bảo toàn trọn vẹn 500.000 bản ghi trước khi tiến hành các bước kiểm toán tiếp theo.

### 2.2.2 Khảo Sát Ngoại Lai Bằng Phương Pháp IQR (Tukey Outliers)

* **Mục tiêu:** Sử dụng rào trên Tukey ($\text{Rào trên} = Q3 + 1.5 \times \text{IQR}$) để kiểm tra ngoại lai trên 6 biến định lượng và quyết định phương án xử lý phù hợp.
* **6 biến số:** `Age`, `AnnualIncome`, `NumberOfPurchases`, `TimeSpentOnWebsite`, `CustomerTenureYears`, `LastPurchaseDaysAgo`.

In [6]:
cols = ['Age', 'AnnualIncome', 'NumberOfPurchases', 'TimeSpentOnWebsite', 'CustomerTenureYears', 'LastPurchaseDaysAgo']
q1, q3 = raw0[cols].quantile(0.25), raw0[cols].quantile(0.75)
rao_tren = (q3 + 1.5 * (q3 - q1)).round(2)

pd.DataFrame({
    'Rao_tren_IQR': rao_tren,
    'Max_thuc_te': raw0[cols].max().round(2),
    'So_luong_vuot': (raw0[cols] > rao_tren).sum(),
    'Ty_le_%': ((raw0[cols] > rao_tren).mean() * 100).round(2)
})

,Rao_tren_IQR,Max_thuc_te,So_luong_vuot,Ty_le_%
Age,97.50,81.00,0,0.00
AnnualIncome,213124.80,204178.29,0,0.00
NumberOfPurchases,31.00,28.00,0,0.00
TimeSpentOnWebsite,88.77,78.36,0,0.00
CustomerTenureYears,6.64,15.35,24740,4.95
LastPurchaseDaysAgo,238.50,189.00,0,0.00


**Nhận xét & Quyết định xử lý ngoại lai:**

* **5 biến không có ngoại lai (0 bản ghi vượt rào):** `Age` (max 81 tuổi), `AnnualIncome` (max 204k USD), `NumberOfPurchases` (max 28 lần), `TimeSpentOnWebsite` (max 78.4 phút), `LastPurchaseDaysAgo` (max 189 ngày) đều có giá trị lớn nhất nằm an toàn dưới rào trên IQR, phản ánh đúng hành vi tiêu dùng thực tế.
* **Biến thâm niên (`CustomerTenureYears`) có ngoại lai:** 24.740 khách hàng (chiếm 4,95%) có thâm niên từ 6.7 đến 15.4 năm (vượt rào trên 6.64 năm).
* **Quyết định: KHÔNG XÓA nhóm vượt rào này.** Thâm niên lâu năm là khách hàng trung thành thực tế của sàn, nếu máy móc dùng IQR cắt bỏ sẽ loại bỏ nhầm nhóm khách hàng giá trị cao. 

### 2.2.3 Kiểm Toán & Xử Lý Dòng Trùng Lặp (Uniqueness)

**Câu hỏi kiểm toán:** Bộ dữ liệu có tồn tại các bản ghi trùng lặp hoàn toàn trên cả 17 thuộc tính không? Nếu có, đây là lỗi hệ thống nhân đôi bản ghi (Duplicate Logging) hay sự trùng hợp ngẫu nhiên giữa hai khách hàng? Việc loại bỏ có làm thay đổi phân phối dữ liệu hay không?

In [25]:
print(f"Số dòng trùng lặp phát hiện: {raw0.duplicated().sum():,} dòng ({raw0.duplicated().mean()*100:.2f}%)")
print(f"Số lần lặp của các bản ghi trùng: {raw0[raw0.duplicated(keep=False)].value_counts().unique()}")

Số dòng trùng lặp phát hiện: 1,148 dòng (0.23%)
Số lần lặp của các bản ghi trùng: [2]


**Phát hiện:** Có 1.148 bản ghi trùng lặp hoàn toàn trên cả 17 cột (lặp đúng 2 lần)chuyển sang kiểm tra mức độ ảnh hưởng lên phân phối dữ liệu.

In [29]:
truoc = raw0.mean(numeric_only=True)
sau = raw0.drop_duplicates().mean(numeric_only=True)

print(((sau - truoc) / truoc * 100).round(4))

Age                     0.0079
AnnualIncome            0.0019
NumberOfPurchases       0.0265
TimeSpentOnWebsite      0.0280
CustomerTenureYears     0.0870
LastPurchaseDaysAgo    -0.0713
LoyaltyProgram          0.0065
DiscountsAvailed        0.0003
SessionCount            0.0051
CustomerSatisfaction    0.0005
PurchaseStatus          0.0648
dtype: float64


**Đánh giá tác động:** Mức lệch trung bình trên toàn bộ 10 biến đặc trưng đều dưới `0,1%` (lớn nhất `+0,0870%` và `-0,0713%`), cho thấy các dòng trùng phân bố ngẫu nhiên và không gây lệch mẫu khi xóa.

In [31]:
raw = raw0.drop_duplicates().reset_index(drop=True)
print(f"Sau khử trùng: {raw.shape}| Dòng trùng còn lại: {raw.duplicated().sum()}")

Sau khử trùng: (498852, 17)| Dòng trùng còn lại: 0


> **Kết luận Mục 2.2.3:** Đã xóa 1.148 dòng trùng, dữ liệu còn 498.852 dòng × 17 cột (`0` dòng trùng lặp), sẵn sàng cho bước kiểm toán 7 quy tắc ở Mục 2.3.

## 2.3 Thiết Lập Bảng Kiểm Toán (Audit Trail) Cho 7 Vấn Đề Nghiệp Vụ

Tuân thủ nguyên tắc quản trị dữ liệu, quy trình làm sạch sử dụng cơ chế **Lưu vết kiểm toán (Audit Trail)** thay vì xóa trực tiếp dữ liệu thô. Bảng cờ lỗi `issues` được khởi tạo song song với 498.852 bản ghi sau khử trùng lặp để ghi nhận độc lập các vi phạm qua 7 quy tắc nghiệp vụ, đảm bảo tính minh bạch và khả năng truy vết lỗi trước khi lọc dữ liệu sạch.

In [10]:
# Khởi tạo bảng lưu vết kiểm toán lỗi cho 500.000 bản ghi gốc
issues = pd.DataFrame(index=raw.index)

### 2.3.1 Vấn đề 1: Sai miền giá trị (Số âm đơn biến)

* **Thuộc tính kiểm toán:** `NumberOfPurchases` (lần), `TimeSpentOnWebsite` (phút), `CustomerTenureYears` (năm), `LastPurchaseDaysAgo` (ngày).
* **Mâu thuẫn logic:** Theo thực tế vật lý và logic nghiệp vụ, tần suất hành động và thời lượng tương tác bắt buộc phải không âm: $\text{Giá trị} \ge 0$. Điều kiện vi phạm: $\text{Giá trị} < 0$.
* **Bản chất nghiệp vụ:** Giá trị `-1` ở số lần mua là mã lỗi kỹ thuật (Error Code) trả về từ cổng thanh toán khi giao dịch bị gián đoạn; trong khi thời gian âm phát sinh do lệch xung nhịp đồng hồ (Clock Skew) giữa các máy chủ ghi log.
* **Xử lý kiểm toán:** Phát hiện **26.947 bản ghi vi phạm (5,40%)**. Xử lý hoàn toàn khách quan (không xét nhãn `PurchaseStatus`) và đánh dấu cờ lỗi vào bảng `issues` thay vì dùng hàm `abs()` vì sẽ làm sai lệch bản chất lỗi kỹ thuật.

In [11]:
# Vấn đề 1: Đánh dấu các bản ghi có giá trị âm ở 4 biến định lượng
issues['neg_values'] = (raw[['NumberOfPurchases', 'TimeSpentOnWebsite', 'CustomerTenureYears', 'LastPurchaseDaysAgo']] < 0).any(axis=1)

### 2.3.2 Vấn đề 2: Nghịch lý thời gian — Mua trước khi tạo tài khoản

* **Thuộc tính kiểm toán:** `CustomerTenureYears` (thâm niên tài khoản, năm) và `LastPurchaseDaysAgo` (số ngày từ lần mua gần nhất).
* **Mâu thuẫn logic:** Tổng số ngày tối đa một tài khoản tồn tại là $T_{\text{max}} = \text{CustomerTenureYears} \times 365.25$ ngày. Điều kiện vi phạm trật tự thời gian: $\text{LastPurchaseDaysAgo} > T_{\text{max}}$.
* **Bản chất & Dẫn chứng:** Khoảng cách từ lần mua gần nhất đến nay lớn hơn tổng thời gian tài khoản được tạo lập — tức **giao dịch mua hàng diễn ra trước cả ngày mở tài khoản**. Điển hình có tài khoản mới tạo 7 ngày (`CustomerTenureYears = 0.02`), nhưng ghi nhận đơn mua từ 180 ngày trước. Nguyên nhân do hệ thống tự động gộp lịch sử mua của khách vãng lai (Guest Checkout) vào tài khoản mới mở mà không đồng bộ lại mốc thời gian đăng ký.
* **Xử lý kiểm toán:** Phát hiện **39.992 bản ghi vi phạm (8,02%)**. Loại bỏ khỏi tập dữ liệu sạch để tránh sai lệch trật tự dòng thời gian của các phiên giao dịch.

In [12]:
# Vấn đề 2: Mua hàng trước ngày tạo tài khoản
issues['purchase_before_tenure'] = raw['LastPurchaseDaysAgo'] > (raw['CustomerTenureYears'] * 365.25)

### 2.3.3 Vấn đề 3: Chưa mua hàng nhưng có ngày mua gần nhất

* **Thuộc tính kiểm toán:** `NumberOfPurchases` (tổng số lần mua hàng) và `LastPurchaseDaysAgo` (số ngày kể từ lần mua gần nhất).
* **Mâu thuẫn logic:** Điều kiện vi phạm: $\text{NumberOfPurchases} = 0 \text{ và } \text{LastPurchaseDaysAgo} > 0$.
* **Bản chất nghiệp vụ:** Xung đột trực tiếp giữa biến đếm tích lũy và biến mốc thời gian sự kiện. Nếu khách hàng chưa từng phát sinh bất kỳ đơn hàng nào trong lịch sử ($N = 0$), thuộc tính "ngày mua gần nhất" bắt buộc phải bằng 0 hoặc không xác định. Việc tồn tại một số ngày cụ thể (ví dụ: đã mua cách đây 45 ngày) khi tổng số đơn bằng 0 là mâu thuẫn nội tại của dữ liệu ghi nhận.
* **Xử lý kiểm toán:** Phát hiện **7.978 bản ghi vi phạm (1,60%)**. Loại bỏ để đảm bảo tính nhất quán nội tại giữa các trường dữ liệu giao dịch.

In [13]:
# Vấn đề 3: Chưa mua hàng nhưng có ngày mua gần nhất
issues['zero_purchases_has_last_date'] = (raw['NumberOfPurchases'] == 0) & (raw['LastPurchaseDaysAgo'] > 0)

### 2.3.4 Vấn đề 4: Chưa mua hàng nhưng đã tiêu dùng mã giảm giá

* **Thuộc tính kiểm toán:** `NumberOfPurchases` (tổng số lần mua) và `DiscountsAvailed` (số mã giảm giá đã sử dụng thành công).
* **Mâu thuẫn logic:** Điều kiện vi phạm: $\text{NumberOfPurchases} = 0 \text{ và } \text{DiscountsAvailed} > 0$.
* **Bản chất nghiệp vụ:** Thuộc tính `DiscountsAvailed` ghi nhận số voucher đã được khấu trừ vào các giao dịch thanh toán thành công (khác với voucher lưu trong ví). Một tài khoản có 0 đơn hàng thành công không thể có lịch sử tiêu dùng thành công từ 1 đến nhiều voucher. Lỗi phát sinh do dịch vụ khuyến mãi đồng bộ nhầm cả voucher bấm áp dụng thử hoặc voucher đã hết hạn vào chỉ số tiêu dùng thực tế.
* **Xử lý kiểm toán:** Phát hiện **7.392 bản ghi vi phạm (1,48%)**. Loại bỏ để phản ánh chính xác hành vi tiêu dùng ưu đãi thực tế của khách hàng.

In [14]:
# Vấn đề 4: Chưa mua hàng nhưng có lượt áp dụng mã giảm giá
issues['zero_purchases_used_discounts'] = (raw['NumberOfPurchases'] == 0) & (raw['DiscountsAvailed'] > 0)

### 2.3.5 Vấn đề 5: Chưa mua hàng nhưng xếp hạng VIP/Premium

* **Thuộc tính kiểm toán:** `NumberOfPurchases` (tổng số lần mua) và `CustomerSegment` (phân hạng CRM: Regular, Premium, VIP).
* **Mâu thuẫn logic:** Điều kiện vi phạm: $\text{NumberOfPurchases} = 0 \text{ và } \text{CustomerSegment} \in [\text{'VIP'}, \text{'Premium'}]$.
* **Bản chất nghiệp vụ:** Khách hàng VIP/Premium có thể mua ít lần nhưng giá trị đơn hàng rất lớn, điều đó hợp lệ; tuy nhiên, với một tài khoản mới đăng ký **chưa từng phát sinh bất kỳ giao dịch nào trong đời ($N = 0$)**, phân khúc mặc định bắt buộc phải là phổ thông (`Regular`). Việc tài khoản $N=0$ mang nhãn VIP/Premium là lỗi gán nhãn phân khúc mặc định từ hệ thống CRM khi đồng bộ hồ sơ.
* **Xử lý kiểm toán:** Phát hiện **6.302 bản ghi vi phạm (1,26%)**. Loại bỏ để bảo đảm tệp phân khúc khách hàng cao cấp phản ánh đúng những tài khoản có đóng góp giá trị thực tế.

In [15]:
# Vấn đề 5: Chưa mua hàng nhưng xếp hạng VIP/Premium
issues['zero_purchases_vip_segment'] = (raw['NumberOfPurchases'] == 0) & (raw['CustomerSegment'].isin(['VIP', 'Premium']))

### 2.3.6 Vấn đề 6: Vi phạm tính hợp lý dân sự — Tuổi mở tài khoản dưới 13 tuổi

* **Thuộc tính kiểm toán:** `Age` (tuổi hiện tại), `CustomerTenureYears` (thâm niên tài khoản).
* **Mâu thuẫn logic:** Độ tuổi tại thời điểm đăng ký tài khoản: $\text{Age}_{\text{mở nick}} = \text{Age} - \text{CustomerTenureYears}$. Điều kiện vi phạm: $\text{Age}_{\text{mở nick}} < 13$.
* **Bản chất & Dẫn chứng:** Theo điều khoản dịch vụ (Terms of Service) phổ quát của các nền tảng thương mại điện tử, độ tuổi tối thiểu để tự đứng tên mở tài khoản giao dịch độc lập là **từ đủ 13 tuổi trở lên**. Trong dữ liệu, nhóm này có độ tuổi lúc mở tài khoản chỉ từ 2 đến dưới 13 tuổi (trung bình 10,9 tuổi), nhưng lại nhiều lần ghi nhận mức **thu nhập độc lập lên đến 50.000 – 100.000 USD/năm**. Đây là lỗi phi lý phát sinh do ghép sai lệch thâm niên với độ tuổi thanh thiếu niên khi đồng bộ dữ liệu.
* **Xử lý kiểm toán:** Phát hiện **3.856 bản ghi vi phạm (0,77%)**. Loại bỏ để bảo toàn tính chân thực về nhân khẩu học và năng lực hành vi dân sự của tập mẫu nghiên cứu.

In [16]:
# Vấn đề 6: Độ tuổi khi tạo tài khoản dưới 13 tuổi
issues['underage_account_creation'] = (raw['Age'] - raw['CustomerTenureYears']) < 13

### 2.3.7 Vấn đề 7: Nghi vấn Bot / Spam đánh giá cực đoan không mua hàng

* **Thuộc tính kiểm toán:** `TimeSpentOnWebsite` (thời gian truy cập, phút), `PurchaseStatus` (0: Không mua, 1: Có mua) và `CustomerSatisfaction` (điểm đánh giá 1–5 sao).
* **Mâu thuẫn logic:** Điều kiện vi phạm: $\text{TimeSpentOnWebsite} < 1.0 \text{ và } \text{PurchaseStatus} = 0 \text{ và } \text{CustomerSatisfaction} \in [1, 5]$.
* **Bản chất hành vi:** Khách hàng vào trang chưa đầy 1 phút, hoàn toàn không phát sinh giao dịch, nhưng lập tức đánh giá ở hai thái cực cảm xúc tuyệt đối: **5 sao** (kịch bản bot seeding ảo) hoặc **1 sao** (review bombing phá hoại). *(Các điểm trung tính 2, 3, 4 sao được giữ lại vì là hành vi tự nhiên của người thật bấm nhanh để tắt khảo sát).*
* **Xử lý kiểm toán:** Phát hiện **592 bản ghi vi phạm (0,12%)**. Đây là trường hợp duy nhất có kết hợp điều kiện `PurchaseStatus == 0`. Do tỷ lệ cực nhỏ (chỉ 223 dòng bị loại riêng bởi lỗi này, làm tỷ lệ mua chỉ lệch 0,02 điểm phần trăm), việc loại bỏ vừa khử được nhiễu bot vừa kiểm soát an toàn rủi ro rò rỉ nhãn.

In [17]:
# Vấn đề 7: Nghi vấn Bot/Spam đánh giá cực đoan không mua hàng
issues['bot_extreme_review'] = (raw['TimeSpentOnWebsite'] < 1) & (raw['PurchaseStatus'] == 0) & (raw['CustomerSatisfaction'].isin([1, 5]))

## 2.4 Bóc Tách Dữ Liệu Sạch & Tổng Hợp Kiểm Toán

**Mục đích:** Hợp nhất các bản ghi vi phạm (toán tử logic OR) để lọc lấy tập dữ liệu sạch (`clean`) thỏa mãn toàn bộ 7 quy tắc, dùng cho phân tích và mô hình hóa.


In [32]:
issues['TONG_LOI_HOP_NHAT'] = issues.any(axis=1)
clean = raw[~issues['TONG_LOI_HOP_NHAT']].reset_index(drop=True)

print(pd.DataFrame({'so_dong': issues.sum(), 'ty_le_%': (issues.mean() * 100).round(2)}))
print(f"\nSạch: {len(clean):,} dòng ({len(clean)/len(raw)*100:.2f}%) | 0 ô thiếu")

                               so_dong  ty_le_%
neg_values                       26947     5.40
purchase_before_tenure           39992     8.02
zero_purchases_has_last_date      7978     1.60
zero_purchases_used_discounts     7392     1.48
zero_purchases_vip_segment        6302     1.26
underage_account_creation         3856     0.77
bot_extreme_review                 592     0.12
TONG_LOI_HOP_NHAT                66596    13.35

Sạch: 432,256 dòng (86.65%) | 0 ô thiếu


### 2.4.1 Chuẩn Hóa Số Liệu Số Thực

Làm tròn 2 chữ số thập phân cho 3 trường số thực liên tục để chuẩn hóa lưu trữ, bảo toàn nguyên vẹn 17 cột gốc.

In [19]:
float_cols = ['AnnualIncome', 'TimeSpentOnWebsite', 'CustomerTenureYears']
clean[float_cols] = clean[float_cols].round(2)
print('Kích thước tập Clean:', clean.shape)

Kích thước tập Clean: (432256, 17)


### 2.4.2 So Sánh Dữ Liệu Trước và Sau Làm Sạch (Before vs After)

**Mục đích:** Đối chiếu trung bình và tỷ lệ biến động (`%`) trên toàn bộ 10 biến số đặc trưng (không nhìn biến mục tiêu `PurchaseStatus`) giữa tập gốc (`raw0`) và tập sạch (`clean`) để kiểm tra sai lệch tổng thể sau khi lọc dữ liệu.

In [20]:
truoc = raw0.drop(columns='PurchaseStatus').mean(numeric_only=True)
sau = clean.drop(columns='PurchaseStatus').mean(numeric_only=True)

pd.DataFrame({'Truoc': truoc, 'Sau': sau, 'Lech_%': (sau - truoc) / truoc * 100}).round(2)

,Truoc,Sau,Lech_%
Age,43.94,44.15,0.48
AnnualIncome,85065.97,85018.02,-0.06
NumberOfPurchases,11.39,11.59,1.79
TimeSpentOnWebsite,30.59,31.15,1.82
CustomerTenureYears,2.16,2.30,6.36
LastPurchaseDaysAgo,60.20,57.58,-4.35
LoyaltyProgram,0.50,0.51,1.16
DiscountsAvailed,3.15,3.15,-0.05
SessionCount,2.35,2.35,-0.05
CustomerSatisfaction,3.22,3.22,-0.01


**Kết luận:**
* **Không gây thiên lệch mẫu tổng thể:** Dù đã loại **67.744 dòng lỗi (13,55%)**, 8/10 biến đặc trưng (`Age`, `AnnualIncome`, `NumberOfPurchases`, `TimeSpentOnWebsite`, `LoyaltyProgram`, `DiscountsAvailed`, `SessionCount`, `CustomerSatisfaction`) đều biến động dưới `2%` (nhóm nhân khẩu học và đánh giá dưới `0,5%`).
* **Thay đổi tập trung đúng vào nhóm biến bị lọc lỗi:** Chỉ `CustomerTenureYears` và `LastPurchaseDaysAgo` lệch đáng kể do trực tiếp loại bỏ các bản ghi mang giá trị âm và nghịch lý thời gian (`LastPurchaseDaysAgo > CustomerTenureYears * 365.25`).
* **Đủ điều kiện đưa vào bước sau:** Tập `clean` (432.256 dòng) đã sạch lỗi kỹ thuật/logic mà không làm biến dạng phân phối nền để dùng cho SQL, thống kê và mô hình hóa.

## 2.5 Xuất Tệp Dữ Liệu & Kiểm Tra Toàn Vẹn

Ghi dữ liệu sạch ra tệp CSV (`utf-8-sig`), sau đó đọc lại từ đĩa để xác nhận toàn vẹn.

In [33]:
clean.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')

chk = pd.read_csv(CLEAN_PATH)
print('Tập sạch (RAM -> File):', clean.shape, '->', chk.shape)
print('Ô thiếu / Trùng lặp   :', chk.isnull().sum().sum(), '/', chk.duplicated().sum())

Tập sạch (RAM -> File): (432256, 17) -> (432256, 17)
Ô thiếu / Trùng lặp   : 0 / 0


---
*Notebook tiếp theo: [3. Data Analysis with SQL](./3.%20Data%20Analysis%20with%20SQL.ipynb)*
